# 01 · LiDAR core — test tile

Phase 1 on one small tile before scaling (brief: test ≤1 km² first).

**Test tile:** DNR portal box on lower Whatcom Creek (~0.1 km²), portal dataset id 1734 (expected: Cascades North 2023, *verify* below).

**Steps:** download → header QA (CRS, density, flight dates, classes) → reproject to EPSG:32610 metres → DTM → normalize → 1 m CHM → 10/20 m metrics on the S2 grid → tree tops.

**Outputs:** `figures/01_test_*.csv`, `figures/01_test_chm.png`; rasters in `DATA/derived/`.

In [ ]:
# --- Setup -------------------------------------------------------------------
REPO_URL  <- "https://github.com/MegaDeadCowboy/forest-height-s2-lidar.git"
REPO_NAME <- "forest-height-s2-lidar"
if (dir.exists("/content") && !dir.exists(file.path("/content", REPO_NAME)))
  system(paste("git clone --depth 1", REPO_URL, file.path("/content", REPO_NAME)))
if (dir.exists("/content")) setwd(file.path("/content", REPO_NAME)) else if (basename(getwd()) == "notebooks") setwd("..")
system("git pull -q")

source("R/setup.R"); source("R/config.R"); source("R/lidar.R")
setup_packages(c("lidR", "terra", "sf", "dplyr", "ggplot2", "readr", "httr2"))

DATA <- if (dir.exists("/content")) "/content/data" else "data"   # single data root (ephemeral on Colab; re-runs re-download)
TILE <- "test_whatcom_falls"
dir.create("figures", showWarnings = FALSE)

## 1 · Download
If the portal refuses the scripted request, download the box in your browser, then in Colab's **Files** panel upload the zip as `/content/data/raw/test_whatcom_falls/download.zip` (create the folders) and re-run this cell — it skips the download when that file exists.

In [ ]:
TEST_URL <- "https://lidarportal.dnr.wa.gov/download?geojson=%7B%22type%22%3A%22Polygon%22%2C%22coordinates%22%3A%5B%5B%5B-122.4373%2C48.7510%5D%2C%5B-122.4373%2C48.7529%5D%2C%5B-122.4304%2C48.7529%5D%2C%5B-122.4304%2C48.7510%5D%2C%5B-122.4373%2C48.7510%5D%5D%5D%7D&ids=1734"

files <- fetch_portal_las(TEST_URL, file.path(DATA, "raw", TILE))
cat("All files downloaded:\n"); print(list.files(file.path(DATA, "raw", TILE), recursive = TRUE))
cat("\nLAS/LAZ:\n"); print(data.frame(file = basename(files), MB = round(file.size(files) / 1e6, 1)))
stopifnot("No LAS/LAZ in the download — check the file list above" = length(files) > 0)

## 2 · Header QA (native CRS)
This fills the Cascades North row in `00_coverage` — every value read from the file, not the PDF.

In [ ]:
hdr <- lidR::readLASheader(files[1])
crs_native <- sf::st_crs(hdr)
print(hdr)
cat("\nNative CRS:", crs_native$Name, "| EPSG:", crs_native$epsg, "| units:", crs_native$units_gdal, "\n")

In [ ]:
las0 <- lidR::readLAS(files, select = "xyzrnc t", filter = "-drop_withheld")
if (is.na(sf::st_crs(las0))) stop("No CRS in LAS header — tell Claude before assigning one.")

hull_m2 <- as.numeric(sf::st_area(sf::st_transform(sf::st_as_sfc(sf::st_bbox(las0)), CRS_EPSG)))   # portal box = rectangle
d <- las0@data
utc <- gps_to_utc(d$gpstime)

header_qa <- tibble::tibble(
  files = length(files), points = nrow(d), area_m2 = round(hull_m2),
  all_pts_m2 = round(nrow(d) / hull_m2, 1),
  first_pts_m2 = round(sum(d$ReturnNumber == 1) / hull_m2, 1),
  ground_pts_m2 = round(sum(d$Classification == 2) / hull_m2, 1),
  crs_name = crs_native$Name, crs_epsg = crs_native$epsg, units = crs_native$units_gdal,
  flown_min = format(min(utc, na.rm = TRUE), "%Y-%m-%d"), flown_max = format(max(utc, na.rm = TRUE), "%Y-%m-%d"))
readr::write_csv(header_qa, "figures/01_test_header_qa.csv")
print(header_qa, width = Inf)

dates <- tibble::tibble(date = as.Date(utc)) |> dplyr::count(date) |>
  dplyr::mutate(pct = round(100 * n / sum(n), 1),
                season = ifelse(as.integer(format(date, "%m")) %in% 5:10, "leaf-on", "leaf-off"))
readr::write_csv(dates, "figures/01_test_flight_dates.csv"); print(dates)

classes <- dplyr::count(d, Classification) |> dplyr::mutate(pct = round(100 * n / sum(n), 2))
readr::write_csv(classes, "figures/01_test_class_counts.csv"); print(classes)

## 3 · Reproject to EPSG:32610, heights to metres
XY via `st_transform`; Z converted separately if the native units are US feet (DNR delivers NAVD88 ft).

In [ ]:
las <- lidR::filter_poi(las0, !Classification %in% c(7L, 18L))   # drop noise
rm(las0); invisible(gc())
las <- sf::st_transform(las, sf::st_crs(CRS_EPSG))

z_in_ft <- grepl("foot|feet|ft", crs_native$units_gdal, ignore.case = TRUE)
if (z_in_ft) {
  las@data[["Z"]] <- las@data[["Z"]] * US_FT
  lidR::las_quantize(las)
  las <- lidR::las_update(las)
}
cat("Z converted from US ft:", z_in_ft, "| Z range (m):", paste(round(range(las$Z), 1), collapse = " – "), "\n")

## 4 · DTM, normalize, 1 m CHM

In [ ]:
t1  <- s2_template(las, res = 1)     # 1 m grid nested in the 10/20 m S2 grid
dtm <- lidR::rasterize_terrain(las, res = t1, algorithm = lidR::tin())
nlas <- lidR::normalize_height(las, lidR::tin())

neg <- sum(nlas$Z < -1); tall <- sum(nlas$Z > 90)
cat("Normalized points < -1 m:", neg, " | > 90 m:", tall, "(dropped)\n")
nlas <- lidR::filter_poi(nlas, Z >= -1, Z <= 90)

chm <- lidR::rasterize_canopy(nlas, res = t1,
                              algorithm = lidR::pitfree(thresholds = c(0, 2, 5, 10, 15), max_edge = c(0, 1.5)))
chm[chm < 0] <- 0

dir.create(file.path(DATA, "derived"), recursive = TRUE, showWarnings = FALSE)
terra::writeRaster(dtm, file.path(DATA, "derived", paste0(TILE, "_dtm_1m.tif")), overwrite = TRUE)
terra::writeRaster(chm, file.path(DATA, "derived", paste0(TILE, "_chm_1m.tif")), overwrite = TRUE)
print(summary(terra::values(chm)))

## 5 · Structure metrics on the Sentinel-2 grid (10 m and 20 m)
Percentiles over returns > 2 m; cover = share of first returns > 2 m. `valid_frac` = share of the cell with CHM data — edge cells under ~0.9 get masked later.

In [ ]:
metric_grid <- function(res) {
  m  <- lidR::pixel_metrics(nlas, ~stand_metrics(Z, ReturnNumber), res = s2_template(las, res))
  vf <- terra::aggregate(!is.na(chm), fact = res, fun = mean)   # chm grid shares the 20 m snap
  m$valid_frac <- terra::extend(terra::crop(vf, m), m)
  m
}
m10 <- metric_grid(10); m20 <- metric_grid(20)
terra::writeRaster(m10, file.path(DATA, "derived", paste0(TILE, "_metrics_10m.tif")), overwrite = TRUE)
terra::writeRaster(m20, file.path(DATA, "derived", paste0(TILE, "_metrics_20m.tif")), overwrite = TRUE)

msum <- terra::as.data.frame(m10, na.rm = TRUE) |>
  dplyr::filter(valid_frac >= 0.9) |>
  dplyr::summarise(cells_10m = dplyr::n(), dplyr::across(c(p50, p95, zmax, cover),
                   list(mean = ~round(mean(.x), 2), sd = ~round(sd(.x), 2))))
readr::write_csv(msum, "figures/01_test_metrics_10m_summary.csv"); print(msum, width = Inf)

## 6 · Tree tops
Variable-window local maxima on the CHM. Window function is a starting value — tune on this tile by eye before scaling.

In [ ]:
ws_fun <- function(h) pmax(3, 0.07 * h + 2)
ttops <- lidR::locate_trees(chm, lidR::lmf(ws = ws_fun, hmin = 5))
area_ha <- sum(!is.na(terra::values(chm))) / 1e4
cat(sprintf("Tree tops (>= 5 m): %d  |  CHM area: %.2f ha  |  stems/ha: %.0f\n",
            nrow(ttops), area_ha, nrow(ttops) / area_ha))

In [ ]:
chm_df <- terra::as.data.frame(chm, xy = TRUE, na.rm = TRUE); names(chm_df)[3] <- "h"
p <- ggplot2::ggplot() +
  ggplot2::geom_raster(data = chm_df, ggplot2::aes(x, y, fill = h)) +
  ggplot2::geom_sf(data = ttops, size = 0.4, colour = "white") +
  ggplot2::scale_fill_viridis_c(name = "Height (m)") +
  ggplot2::coord_sf(crs = CRS_EPSG, datum = CRS_EPSG) +
  ggplot2::labs(title = "1 m canopy height model, test tile (lower Whatcom Creek)",
                caption = sprintf("WA DNR lidar, pit-free CHM · %d tree tops · EPSG:%d", nrow(ttops), CRS_EPSG)) +
  ggplot2::theme_minimal(base_size = 9)
ggplot2::ggsave("figures/01_test_chm.png", p, width = 8, height = 4.5, dpi = 200, bg = "white")
p

## 7 · Check before scaling
Paste back: header QA, flight dates, class counts, Z range line, CHM summary, metrics summary, tree count, and the figure.
- [ ] Project/dates confirm this is Cascades North 2023 (or say what it is)
- [ ] Leaf-on vs leaf-off share recorded
- [ ] CHM looks right: no pits/spikes, stream corridor and gaps visible, heights plausible
- [ ] Tree tops sit on crowns (tune `ws_fun` if not)